<a href="https://colab.research.google.com/github/umeshgurjar914042-creator/rag-chatbot/blob/main/rag_basics_day8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!pip install google-genai numpy

In [13]:
from google import genai
from google.colab import userdata
import numpy as np

# API की से सभी न्यूलाइन(\n) और कैरिज रिटर्न(\r) कैरेक्टर्स को पूरी तरह हटाएं
api_key = userdata.get("GEMINI_API_KEY")
if api_key:
    api_key = api_key.replace("\r", "").replace("\n", "").strip()

client = genai.Client(api_key=api_key)

In [15]:
# दस्तावेज़ के टुकड़े (Chunks) तैयार करना
डॉक्यूढेंट = """कंपनी की छुट्टी नीतरी: हर कर्मचारी को साल में 12 आकस्मिक छुट्टियाँ मिलती हूँ।
बीमारी की छुट्टी 10 दिन की होती है और उसके लिए डॉक्टर का पर्चा जरूरी है।
कर्मचारी वर्क फ्रॉम होम हफ्ते में 2 दिन कर सकते हैं।
वेतन हर महीने की 5 तारीख को खाते में आता है।

ऒफिस का समय सुबह 10 बजे से शाम 6 बजे तक है।"""

chunks = [l.strip() for l in डॉक्यूढेंट.split("\n") if l.strip()]

# text-embedding-004 का उपयोग करके एम्बेडिंग्स जेनरेट करना
response = client.models.embed_content(
    model="text-embedding-004",
    contents=chunks
)

embeddings = [e.values for e in response.embeddings]

print(f"Generated {len(embeddings)} embeddings.")
print(f"Dimension of each embedding: {len(embeddings[0])}")

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

In [16]:
def मिलान(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def खोजो(सवाल):
    # text-embedding-004 का उपयोग करके सवाल का एम्बेडिंग जेनरेट करें
    response = client.models.embed_content(
        model="text-embedding-004",
        contents=सवाल
    )
    q = response.embeddings[0].values

    # सभी चंक्स के साथ कोसाइन सिमिलैरिटी की तुलना करें
    scores = [मिलान(q, e) for e in embeddings]
    return chunks[int(np.argmax(scores))]

print(खोजो("वेतन कब मिलता है?"))

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

In [ ]:
def जवाब(सवाल):
    संदर्भ = खोजो(सवाल)
    प्रॉम्प्ट = f"""नीचे दिए संदर्भ के आधार पर ही सवाल का जल्दी जवाब दोॢ
अगर संदर्भ में जवाब न हो तो कहो "मुछे नहीं पता"।

संदर्भ: {संदर्भ}

सवाल: {सवाल}"""
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=प्रॉम्प्ट
    )
    return response.text

print(जवाब("गर से काम कितने दिन कर सकते हैं?"))
print(जवाब("कंपनी में खाना मुफ्त है क्या?"))

In [17]:
def खोजो_k(सवाल, k=2):
    # text-embedding-004 मॉडल का उपयोग करके क्वेरी एम्बेड करें
    response = client.models.embed_content(
        model="text-embedding-004",
        contents=सवाल
    )
    q = response.embeddings[0].values
    scores = [मिलान(q, e) for e in embeddings]
    top = np.argsort(scores)[::-1][:k]
    return [chunks[i] for i in top]

print(खोजो_k("छुट्टियों के बारे में बताओ", k=2))

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

In [ ]:
def जवाब_k(सवाल, k=2):
    संदर्भ = "\n".join(खोजो_k(सवाल, k))
    प्रॉम्प्ट = f"""नीचे दिए संदर्भ के आधार पर ही सवाल का जवाब दो।
अगर संदर्भ में जवाब न हो तो कहो "मुझे नहीं पता"।

संदर्भ:
{संदर्भ}

सवाल: {सवाल}"""
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=प्रॉम्प्ट
    )
    return response.text

print(जवाब_k("कुल कितनी छुट्टियाँ मिल सकती हैं?"))

In [18]:
while True:
    सवाल = input("आपका सवाल (बंद करने के लिए 'exit'): ")
    if सवाल.strip().lower() == "exit":
        break
    print(जवाब_k(सवाल))
    print("-" * 30)

KeyboardInterrupt: Interrupted by user

In [19]:
import numpy as np

# 1. सबसे पहले एम्बेडिंग्स जेनरेट करें
try:
    response = client.models.embed_content(
        model="text-embedding-004",
        contents=chunks
    )
    embeddings = [e.values for e in response.embeddings]
    print("एम्बेडिंग्स सफलतापूर्वक जेनरेट हो गईं!")
except Exception as e:
    print("त्रुटि:", e)


त्रुटि: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}


In [20]:
# 2. टेस्ट केस चलाएं और स्कोर देखें
टेस्ट = [
    ("बीमारी की छुट्टी कितने दिन की है?", "10"),
    ("आकस्मिक छुट्टियाँ कितनी मिलती हैं?", "12"),
    ("वेतन किस तारीख को आता है?", "5"),
    ("वर्क फ्रॉम होम कितने दिन?", "2"),
    ("बोनस कितना मिलता है?", "नहीं पता"),
]

सही = 0
for सवाल, उम्मीद in टेस्ट:
    ans = जवाब_k(सवाल)
    ठीक = उम्मीद in ans or ("नहीं पता" in उम्मीद and ("नहीं" in ans or "पता" in ans))
    सही += ठीक
    print("✅" if ठीक else "❌", सवाल, "->", ans.strip()[:80])

print(f"\nस्कोर: {सही}/{len(टेस्ट)}")

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}